## Свёрточные сети для классификации

In [1]:
from typing import Type

import torch
from torch import Tensor, nn
from torch.nn import functional as F

#### Задание 1. Skip-connections (2 балла)

Постройте архитектуру свёрточной сети, аналогичную архитектуре в примере ниже, но добавьте в неё skip-connections, то есть дополнительные рёбра в вычислительном графе, позволяющие пропускать градиент в более ранние слои напрямую, минуя очередной блок Conv2D + BatchNorm + ReLU:

```python
def forward(self, x: Tensor) -> Tensor:
    x = x + self.block1(x)
    x = self.maxpool(x)
    x = x + self.block2(x)
    x = self.maxpool(x)
    ...
    x = x.adaptive_maxpool(x).flatten(1)
    logits = self.fc(x)
    return logits
```


Наша верхнеуровневая архитектура будет выглядеть так:

In [2]:
class MyResNet(nn.Module):
    def __init__(
        self,
        block: Type[nn.Module],
        n_classes: int,
        hidden_channels: list[int] = [32, 64],
    ) -> None:
        super().__init__()
        # входной слой, принимающий изображение с 3-мя каналами
        self.in_conv = nn.Conv2d(3, hidden_channels[0], kernel_size=3, stride=1)
        self.relu = nn.ReLU(inplace=True)

        # собираем свёрточные блоки, каждый задаётся кол-вом входных и выходных каналов
        blocks = []
        for c_in, c_out in zip(hidden_channels[:-1], hidden_channels[1:]):
            # добавляем очередной блок
            blocks.append(block(c_in, c_out))
            # добавляем Max pooling для уменьшения размерности
            blocks.append(nn.MaxPool2d(2, 2))

        # собираем блоки в единый Sequential модуль для удобства
        self.features = nn.Sequential(*blocks)
        self.maxpool = nn.AdaptiveMaxPool2d(1)

        # линейный слой для классификации
        self.fc = nn.Linear(hidden_channels[-1], n_classes)

    def forward(self, x: Tensor) -> Tensor:
        h = self.features(self.relu(self.in_conv(x)))
        logits = self.fc(self.maxpool(h).flatten(1))
        return logits

Базовый блок, без residual connections, состоит из двух свёрток и нормализаций:

In [3]:
class BasicBlock(nn.Module):
    def __init__(self, inplanes: int, planes: int) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(
            inplanes, planes, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(
            planes, planes, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(planes)

    def forward(self, x: Tensor) -> Tensor:
        # first conv + bn + nonlinearity
        out = self.relu(self.bn1(self.conv1(x)))
        # second conv + bn
        out = self.bn2(self.conv2(out))
        # final nonlinearity
        out = self.relu(out)
        return out

Посмотрим на результат его применения к тензору:

In [ ]:
BasicBlock(4, 6).forward(torch.randn(3, 4, 32, 32)).shape

torch.Size([3, 6, 32, 32])

Теперь нужно изменить этот блок, добавив в него skip-connection. Теперь в методе `forward` входной тензор `x` пойдёт по двум веткам:
1. как в базовом блоке, через наши всёртки и нормализации, до последней нелинейности
2. в обход свёрток и нормализаций

В конце эти ветки нужно объединить через сумму. Тут есть проблема: в исходном тензоре `x` и обработанном нашим блоком `h(x)` отличается количество каналов (остальные размерности совпадают). То есть нам нужно сравнять количество каналов исходного тензора `inplanes` с количеством выходных каналов `outplanes`.

Интуитивно, если рассматривать каждый пиксель входного тензора как вектор размера `inplanes`, в вектор размера `planes` его можно превратить домножением на матрицу размера `inplanes x planes`. Это можно сделать, создав свёрточный слой с размером кернела 1 - он и будет переводить наши пиксели в другую размерность.

Не забудьте к сумме каналов применить нелинейность.

In [4]:
class ResidualBlock(nn.Module):
    def __init__(self, inplanes: int, planes: int) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(
            inplanes, planes, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(
            planes, planes, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(planes)

        # добавьте свёртку 1x1 для изменения кол-ва каналов входного тензора
        self.conv11 = nn.Conv2d(inplanes, planes, kernel_size=1, stride=1, bias=False)
        self.bn11 = nn.BatchNorm2d(planes)

    def forward(self, x: Tensor) -> Tensor:
        # сохраним входной тензор на будущее
        identity = x

        # ВАШ ХОД
        out_conv = self.relu(self.bn1(self.conv1(x)))
        out_conv = self.bn2(self.conv2(out_conv))

        out_skip = self.conv11(identity)

        out = out_conv + out_skip
        out = self.relu(out)
        return out

Проверим размеры:

In [ ]:
assert ResidualBlock(4, 6).forward(torch.randn(3, 4, 32, 32)).shape == torch.Size(
    [3, 6, 32, 32]
)

Проверим, что модель выдаёт тензор ожидаемого размера:

In [ ]:
MyResNet(ResidualBlock, 7, hidden_channels=[16, 32, 64, 128]).forward(
    torch.randn(3, 3, 32, 32)
).shape

torch.Size([3, 7])

Теперь мы можем создавать модели разного размера, в том числе достаточно большие и глубокие, чтобы хорошо классифицировать изображения из датасета CIFAR-10.

In [ ]:
sum(
    p.numel()
    for p in MyResNet(ResidualBlock, 7, hidden_channels=[16, 32, 64, 64]).parameters()
)

151367

#### Задание 2. Обучение `MyResNet` с использованием Lightning (5 баллов)

Ваша задача: добиться 80% точности на валидационной выборке с вашей реализацией `MyResNet`.

После окончания обучения используйте метод `Trainer.validate` для вывода ваших метрик с удачного чекпоинта модели.

NB: вызывайте `Trainer.validate` везде, где в задании требуется достичь какой-то точности


Советы:
- По умолчанию Lightning сохраняет только последний чекпоинт, так что вам может потребоваться `lightning.callbacks.ModelCheckpoint`, чтобы сохранять лучший чекпоинт в процессе обучения.

- Чтобы добиться нужной точности, ваша модель должна быть достаточно глубокой, ориентируйтесь на 4-5 блоков.

- Используйте tensorboard, чтобы следить за динамикой обучения. Если заметите переобучение — подключайте регуляризацию. Большая модель с регуляризацией обычно лучше маленькой модели без неё.



In [ ]:
pip install pytorch-lightning torchmetrics

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pytorch_lightning as L
import torch
import torchmetrics
from torch import Tensor, nn
from torch.nn import functional as F

from typing import Callable

from pytorch_lightning.utilities.types import EVAL_DATALOADERS, TRAIN_DATALOADERS, STEP_OUTPUT
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger
from PIL.Image import Image
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)
L.seed_everything(42)

Seed set to 42


42

In [6]:
def create_classification_metrics(
    num_classes: int, prefix: str
) -> torchmetrics.MetricCollection:
    return torchmetrics.MetricCollection(
        [
            torchmetrics.Accuracy(task="multiclass", num_classes=num_classes),
            torchmetrics.classification.MulticlassAUROC(
                num_classes=num_classes, average="macro"
            ),
        ],
        prefix=prefix,
    )

In [7]:
class Datamodule(L.LightningDataModule):
    def __init__(
        self,
        batch_size: int,
        transform: Callable[[Image], Tensor] = transforms.ToTensor(),
        num_workers: int = 15,
    ):
        super().__init__()
        self.batch_size = batch_size
        self.train_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.1,  contrast=0.1, saturation=0.1),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
        ])
        self.val_transform = transforms.ToTensor()
        self.num_workers = num_workers

    def prepare_data(self) -> None:
        pass

    def setup(self, stage: str) -> None:
        if stage == "fit":
            self.train_dataset = datasets.CIFAR10(
                "data",
                train=True,
                download=True,
                transform=self.train_transform,
            )
            self.val_dataset = datasets.CIFAR10(
                "data",
                train=False,
                download=True,
                transform=self.val_transform,
            )
        elif stage == "validate":
            self.val_dataset = datasets.CIFAR10(
                "data",
                train=False,
                download=True,
                transform=self.val_transform,
            )
        else:
            raise NotImplementedError

    def train_dataloader(self) -> TRAIN_DATALOADERS:
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
        )

    def val_dataloader(self) -> EVAL_DATALOADERS:
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
        )

In [8]:
from typing import Any

class Lit(L.LightningModule):
    def __init__(self, model: nn.Module, learning_rate: float) -> None:
        super().__init__()
        self.save_hyperparameters(ignore=['model'])
        self.model = model
        self.learning_rate = learning_rate
        self.train_metrics = create_classification_metrics(num_classes=10, prefix="train_")
        self.val_metrics = create_classification_metrics(num_classes=10, prefix="val_")

    def training_step(
        self, batch: tuple[Tensor, Tensor], batch_idx: int
    ) -> STEP_OUTPUT:

        x, y = batch
        y_hat = self.model(x)
        loss = F.cross_entropy(y_hat, y)
        self.log("train_loss", loss, on_step=False, on_epoch=True)
        self.train_metrics.update(y_hat, y)
        self.log_dict(self.train_metrics, on_step=False, on_epoch=True)
        return loss

    def validation_step(
        self, batch: tuple[Tensor, Tensor], batch_idx: int
    ) -> STEP_OUTPUT | None:
        x, y = batch
        y_hat = self.model(x)
        loss = F.cross_entropy(y_hat, y)
        self.log("val_loss", loss, on_epoch=True, on_step=False)
        self.val_metrics.update(y_hat, y)
        self.log_dict(self.val_metrics, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self) -> dict[str, Any]:
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.learning_rate)
        return {
            "optimizer": optimizer,
            "lr_scheduler": torch.optim.lr_scheduler.MultiStepLR(
                optimizer,
                milestones=[5, 10, 15],
                gamma=0.1,
            ),
        }

In [9]:
from pytorch_lightning.callbacks import Callback, EarlyStopping, ModelCheckpoint


callbacks = [
    ModelCheckpoint(
        filename="{epoch}-{val_MulticlassAccuracy:.4f}",
        monitor="val_MulticlassAccuracy",
        mode="max",
        save_top_k=2,
        save_last=True,
    ),
    EarlyStopping(
        monitor="val_MulticlassAccuracy",
        mode="max",
        patience=2,
        min_delta=0.01
    ),
]

In [ ]:
trainer = L.Trainer(
        max_epochs=20,
        accelerator='auto',
        devices=1,
        callbacks=callbacks,
        logger=TensorBoardLogger(save_dir="task1"),
        enable_progress_bar=True,
    )

lit_module = Lit(
    model=MyResNet( block=ResidualBlock, n_classes=10, hidden_channels=[32, 64, 128, 256]),
    learning_rate=0.001,
)

datamodule = Datamodule(batch_size=64, num_workers=15)

trainer.fit(
    model=lit_module,
    datamodule=datamodule,
    ckpt_path=None
)

model_last_checkpoint = trainer.checkpoint_callback.best_model_path

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3050 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type             | Params | Mode 
-----------------------------------------------------------
0 | model         | MyResNet         | 1.2 M  | train
1 | train_metrics | MetricCollection | 0      | train
2 | val_metrics   | MetricCollection | 0      | train
-----------------------------------------------------------
1.2 M     Trainable params
0         Non-trainable params
1.2 M     Total params
4.842     Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [ ]:
best_model = Lit.load_from_checkpoint(model_last_checkpoint,
                                      model=MyResNet(block=ResidualBlock, n_classes=10, hidden_channels=[32, 64, 128, 256])
)
val_results = trainer.validate(model = best_model, datamodule = datamodule)
val_accuracy = val_results[0]['val_MulticlassAccuracy']
print(f"Accuracy:, {val_accuracy:.4f}")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   val_MulticlassAUROC      0.9930137991905212
 val_MulticlassAccuracy      0.89410001039505
        val_loss            0.3171307146549225
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Accuracy:, 0.8941


Результаты(без использования torchvision.tranforms):

   val_MulticlassAUROC:      0.9930137991905212

 val_MulticlassAccuracy:      0.89410001039505

        val_loss:            0.3171307146549225

Accuracy:, 0.8941


#### Задание 3. Добавление аугментаций (1 балл + 2 балла за точность на валидации более 85%)

Добавьте к обучающему датасету аугментации - случайные трансформации входных данных. Для этого можно использовать `torchvision.transforms` и `albumentations`.

С `torchvision.transforms` совсем просто: вам нужно будет при создании `Datamodule` из практики по `lightning` указать вместо

```python
transform = transforms.ToTensor()
```
композицию трансформаций:

```python
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),  # случайное зеркальное отражение
    ...
    transforms.ToTensor(),
])
```

В пакете `albumentations` аугментаций значительно больше:

![albumentations](https://albumentations.ai/_next/image/?url=%2Fassets%2Ftop_image.webp&w=1080&q=75)

In [ ]:
class AugDatamodule(L.LightningDataModule):
    def __init__(
        self,
        batch_size: int,
        transform: Callable[[Image], Tensor] = transforms.ToTensor(),
        num_workers: int = 15,
    ):
        super().__init__()
        self.batch_size = batch_size
        self.train_transform = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.1,  contrast=0.1, saturation=0.1),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
        ])
        self.val_transform = transforms.ToTensor()
        self.num_workers = num_workers

    def prepare_data(self) -> None:
        pass

    def setup(self, stage: str) -> None:
        if stage == "fit":
            self.train_dataset = datasets.CIFAR10(
                "data",
                train=True,
                download=True,
                transform=self.train_transform,
            )
            self.val_dataset = datasets.CIFAR10(
                "data",
                train=False,
                download=True,
                transform=self.val_transform,
            )
        elif stage == "validate":
            self.val_dataset = datasets.CIFAR10(
                "data",
                train=False,
                download=True,
                transform=self.val_transform,
            )
        else:
            raise NotImplementedError

    def train_dataloader(self) -> TRAIN_DATALOADERS:
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
        )

    def val_dataloader(self) -> EVAL_DATALOADERS:
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
        )

In [ ]:

class AugLit(L.LightningModule):
    def __init__(self, model: nn.Module, learning_rate: float) -> None:
        super().__init__()
        self.save_hyperparameters(ignore=['model'])
        self.model = model
        self.learning_rate = learning_rate
        self.train_metrics = create_classification_metrics(num_classes=10, prefix="train_")
        self.val_metrics = create_classification_metrics(num_classes=10, prefix="val_")

    def training_step(
        self, batch: tuple[Tensor, Tensor], batch_idx: int
    ) -> STEP_OUTPUT:

        x, y = batch
        y_hat = self.model(x)
        loss = F.cross_entropy(y_hat, y)
        self.log("train_loss", loss, on_step=False, on_epoch=True)
        self.train_metrics.update(y_hat, y)
        self.log_dict(self.train_metrics, on_step=False, on_epoch=True)
        return loss

    def validation_step(
        self, batch: tuple[Tensor, Tensor], batch_idx: int
    ) -> STEP_OUTPUT | None:
        x, y = batch
        y_hat = self.model(x)
        loss = F.cross_entropy(y_hat, y)
        self.log("val_loss", loss, on_epoch=True, on_step=False)
        self.val_metrics.update(y_hat, y)
        self.log_dict(self.val_metrics, on_step=False, on_epoch=True)
        return loss

    def configure_optimizers(self) -> dict[str, Any]:
        optimizer = torch.optim.Adam(self.model.parameters(), lr=self.learning_rate)
        return {
            "optimizer": optimizer,
            "lr_scheduler": torch.optim.lr_scheduler.MultiStepLR(
                optimizer,
                milestones=[5, 10, 15],
                gamma=0.1,
            ),
        }

In [ ]:

callbacks = [
    ModelCheckpoint(
        filename="{epoch}-{val_MulticlassAccuracy:.4f}",
        monitor="val_MulticlassAccuracy",
        mode="max",
        save_top_k=2,
        save_last=True,
    ),
    EarlyStopping(
        monitor="val_MulticlassAccuracy",
        mode="max",
        patience=2,
        min_delta=0.01
    ),
]

In [ ]:
aug_trainer = L.Trainer(
        max_epochs=20,
        accelerator='auto',
        devices=1,
        callbacks=callbacks,
        logger=TensorBoardLogger(save_dir="task2"),
        enable_progress_bar=True,
    )

aug_lit_module = AugLit(
    model=MyResNet( block=ResidualBlock, n_classes=10, hidden_channels=[32, 64, 128, 256]),
    learning_rate=0.001,
)

aug_datamodule = AugDatamodule(batch_size=64, num_workers=15)

aug_trainer.fit(
    model=aug_lit_module,
    datamodule=aug_datamodule,
    ckpt_path=None
)

aug_model_last_checkpoint =aug_trainer.checkpoint_callback.best_model_path

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type             | Params | Mode 
-----------------------------------------------------------
0 | model         | MyResNet         | 1.2 M  | train
1 | train_metrics | MetricCollection | 0      | train
2 | val_metrics   | MetricCollection | 0      | train
-----------------------------------------------------------
1.2 M     Trainable params
0         Non-trainable params
1.2 M     Total params
4.842     Total estimated model params size (MB)
39        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [ ]:
aug_best_model = AugLit.load_from_checkpoint(aug_model_last_checkpoint,
                                      model=MyResNet(block=ResidualBlock, n_classes=10, hidden_channels=[32, 64, 128, 256])
)
val_results = aug_trainer.validate(model = aug_best_model, datamodule = aug_datamodule)
val_accuracy = val_results[0]['val_MulticlassAccuracy']
print(f"Accuracy:, {val_accuracy:.4f}")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   val_MulticlassAUROC      0.9925575256347656
 val_MulticlassAccuracy     0.8860999941825867
        val_loss            0.3272870182991028
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Accuracy:, 0.8861


Результаты с использованием torchvision.transforms

   val_MulticlassAUROC      0.9925575256347656

 val_MulticlassAccuracy     0.8860999941825867

        val_loss            0.3272870182991028

Accuracy:, 0.8861

#### Задание 4. Использование предобученной модели (4 балла)

Теперь мы научимся использовать модели, обученные на других задачах

Ваша задача: добиться 90% точности на тестовой выборке CIFAR-10. Постарайтесь уложиться модель с ~5 млн параметров

В `torchvision.models` есть много реализованных архитектур, размером которых можно удобно управлять. Например, ниже можно создать крошечную версию модели `MobileNetV2`:

In [10]:
from torchvision.models import MobileNetV2

mobilenet = MobileNetV2(
    num_classes=10,
    width_mult=0.4,
    inverted_residual_setting=[
        # t, c, n, s
        [1, 16, 1, 1],
        [3, 24, 2, 2],
        [3, 32, 3, 2],
    ],
    dropout=0.2,
)

sum([param.numel() for param in mobilenet.parameters()])

46322

Но кроме архитектуры модели, мы также можем скачать веса, полученные при обучении на каком-то датасете. Например, для нашей задачи можно использовать предобучение на самом известном датасете для классификации изображений - ImageNet:

In [11]:
from torchvision.models.efficientnet import EfficientNet_B0_Weights, efficientnet_b0

# создаём EfficientNet с весами, полученными на ImageNet
weights = EfficientNet_B0_Weights.IMAGENET1K_V1
efficientnet = efficientnet_b0(weights=weights)
sum([param.numel() for param in efficientnet.parameters()])

5288548

**Указание 1.** С использованием модели в исходном виде есть проблема: в ImageNet 1000 классов, а у нас только 10. Поэтому в предобученной модели нужно будет полностью заменить последний линейный слой, который даёт распределение вероятностей классов. Это можно сделать уже в готовом объекте модели, переназначив атрибут.

Подсказка: в `efficientnet_b0` линейный слой находится в атрибуте `classifier`


**Указание 2.** Все слои, кроме нескольких последних (может быть, только последнего) мы можем заморозить, то есть сделать значения параметров в них неизменными. Это позволит и сохранить способность модели выделять полезные низкоуровневые признаки (она научилась этому на ImageNet), и существенно ускорить дообучение.


Чтобы заморозить параметры, нужно всего лишь отключить для них расчёт градиентов. Вернитесь к первой практике, чтобы вспомнить, как это можно сделать. Нам подойдёт самый простой способ с `.requires_grad`.

Подсказка: в `efficientnet_b0` свёрточные слои находятся в атрибуте `features`

**Указание 3.** Предобученные модели на ImageNet ожидают специальным образом трансформированные изображения:


In [12]:
weights.transforms()

ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BICUBIC
)

Поэтому эти трансформации нужно будет передать в датамодуль (как мы делали с аугментациями).

ВАШ ХОД: Обучите модель и выведите результат метода validate на удачном чекпоинте

In [15]:
from torchvision.transforms import InterpolationMode

class EfficientNetDatamodule(L.LightningDataModule):
    def __init__(
        self,
        batch_size: int,
        transform: Callable[[Image], Tensor] = transforms.ToTensor(),
        num_workers: int = 15,
    ):
        super().__init__()
        self.batch_size = batch_size

        wgs = EfficientNet_B0_Weights.IMAGENET1K_V1
        self.weights = wgs.transforms()

        self.train_transform = transforms.Compose([
            transforms.RandomResizedCrop(self.weights.crop_size[0], interpolation=InterpolationMode.BICUBIC),
            #transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.1,  contrast=0.1, saturation=0.1),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            ),
        ])

        self.val_transform = self.weights
        self.num_workers = num_workers

    def prepare_data(self) -> None:
        datasets.CIFAR10('data', train=True, download=True)
        datasets.CIFAR10('data', train=False, download=True)

    def setup(self, stage: str) -> None:
        if stage == "fit":
            self.train_dataset = datasets.CIFAR10(
                "data",
                train=True,
                download=True,
                transform=self.train_transform,
            )
            self.val_dataset = datasets.CIFAR10(
                "data",
                train=False,
                download=True,
                transform=self.val_transform,
            )
        elif stage == "validate":
            self.val_dataset = datasets.CIFAR10(
                "data",
                train=False,
                download=True,
                transform=self.val_transform,
            )
        else:
            raise NotImplementedError

    def train_dataloader(self) -> TRAIN_DATALOADERS:
        return DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
        )

    def val_dataloader(self) -> EVAL_DATALOADERS:
        return DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
        )

In [16]:
class EfficientNetLit(L.LightningModule):
    def __init__(self, learning_rate: float) -> None:
        super().__init__()
        self.save_hyperparameters(ignore=['model'])

        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        self.model = efficientnet_b0(weights=weights)

        self.learning_rate = learning_rate

        in_features = self.model.classifier[1].in_features
        self.model.classifier[1] = nn.Linear(in_features, 10)

        for param in self.model.parameters():
            param.requires_grad = False

        for param in self.model.classifier.parameters():
            param.requires_grad = True

        for param in self.model.features[7].parameters():
            param.requires_grad = True

        for param in self.model.features[8].parameters():
            param.requires_grad = True

        self.train_metrics = create_classification_metrics(num_classes=10, prefix="train_")
        self.val_metrics = create_classification_metrics(num_classes=10, prefix="val_")

    def training_step(
        self, batch: tuple[Tensor, Tensor], batch_idx: int
    ) -> STEP_OUTPUT:

        x, y = batch
        y_hat = self.model(x)
        loss = F.cross_entropy(y_hat, y)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        self.train_metrics.update(y_hat, y)
        self.log_dict(self.train_metrics, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(
        self, batch: tuple[Tensor, Tensor], batch_idx: int
    ) -> STEP_OUTPUT | None:
        x, y = batch
        y_hat = self.model(x)
        loss = F.cross_entropy(y_hat, y)
        self.log("val_loss", loss, on_epoch=True, on_step=False, prog_bar=True)
        self.val_metrics.update(y_hat, y)
        self.log_dict(self.val_metrics, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self) -> dict[str, Any]:
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=self.learning_rate, weight_decay=1e-5)
        return {
            "optimizer": optimizer,
            "lr_scheduler": torch.optim.lr_scheduler.MultiStepLR(
                optimizer,
                milestones=[5, 10, 15],
                gamma=0.1,
            ),
        }

In [17]:
callbacks = [
    ModelCheckpoint(
        filename="{epoch}-{val_MulticlassAccuracy:.4f}",
        monitor="val_MulticlassAccuracy",
        mode="max",
        save_top_k=2,
        save_last=True,
    ),
    EarlyStopping(
        monitor="val_MulticlassAccuracy",
        mode="max",
        patience=2,
        min_delta=0.005
    ),
]

In [18]:

eff_trainer = L.Trainer(
        max_epochs=15,
        accelerator='auto',
        devices=1,
        callbacks=callbacks,
        logger=TensorBoardLogger(save_dir="task2"),
        enable_progress_bar=True,
    )

eff_lit_module = EfficientNetLit(learning_rate=0.001)

eff_datamodule = EfficientNetDatamodule(batch_size=64, num_workers=15)

eff_trainer.fit(
    model=eff_lit_module,
    datamodule=eff_datamodule,
)

eff_model_last_checkpoint = eff_trainer.checkpoint_callback.best_model_path

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3050 Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type             | Params | Mode 
-----------------------------------------------------------
0 | model         | EfficientNet     | 4.0 M  | train
1 | train_metrics | MetricCollection | 0      | train
2 | val_metrics   | MetricCollection | 0      | train
-----------------------------------------------------------
1.1 M     Trainable params
2.9 M     Non-trainable params
4.0 M     Total params
16.081    Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

In [19]:
best_eff_model = EfficientNetLit.load_from_checkpoint(eff_model_last_checkpoint)
eff_val_results = eff_trainer.validate(model = best_eff_model, datamodule = eff_datamodule)
eff_val_accuracy = eff_val_results[0]['val_MulticlassAccuracy']
print(f"Accuracy:, {eff_val_accuracy:.4f}")

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   val_MulticlassAUROC      0.9954938888549805
 val_MulticlassAccuracy     0.9157000184059143
        val_loss            0.2555268406867981
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Accuracy:, 0.9157


Результаты:

   val_MulticlassAUROC:      0.9954938888549805

 val_MulticlassAccuracy:     0.9157000184059143

        val_loss:            0.2555268406867981

Accuracy:, 0.9157